# A Casa que Lembra

Jogo de terror em texto para o **Google Colab**.

## Como usar
1. Faça upload deste arquivo no [Google Colab](https://colab.research.google.com/) (**Arquivo → Fazer upload do notebook**)  
   ou abra pelo Drive.
2. Execute a **célula 1** (código do jogo) — defina as funções.
3. Execute a **célula 2** (`jogar()`) — digite o número da opção quando pedir.
4. Para jogar de novo, rode de novo a célula 2 (ela reinicia o estado).

**Regras:** máximo **15** interações · **3** de vida · vários finais (fuga, verdade, eco, ritual secreto, morte, atrasado).


In [ ]:
# === A Casa que Lembra (versão Colab, arquivo único) ===

MAX_TURNOS = 15

def estado_inicial():
    return {
        "vida": 3,
        "inv": [],
        "turnos": 0,
        "ouviu_fita": False,
        "viu_foto": False,
        "abriu_porao": False,
        "apagou_vela": False,
        "conheceu_eco": False,
        "leu_bilhete": False,
        "viu_tv": False,
        "forcou_porao": False,
    }

state = estado_inicial()


def resetar():
    """Reinicia o jogo (chame antes de jogar de novo)."""
    global state
    state = estado_inicial()


def pegar(item):
    if item not in state["inv"]:
        state["inv"].append(item)


def tem(item):
    return item in state["inv"]


def perder_vida(n=1):
    state["vida"] -= n
    return state["vida"] <= 0


def status_linha():
    itens = ", ".join(state["inv"]) if state["inv"] else "nada"
    return (
        f"[vida: {state['vida']} | turnos: {state['turnos']}/{MAX_TURNOS} "
        f"| inventário: {itens}]"
    )


def narrar(*paragrafos):
    print()
    for p in paragrafos:
        print(p)
        print()


def escolher(msg, opcoes):
    opcoes_norm = [o.strip().lower() for o in opcoes]
    while True:
        print(status_linha())
        try:
            r = input(msg).strip().lower()
        except EOFError:
            print("\n(Entrada encerrada.)")
            state["turnos"] += 1
            return opcoes_norm[0]
        if r in opcoes_norm:
            state["turnos"] += 1
            return r
        print("Opção inválida. Tente de novo.")


# ---------- CENAS ----------

def inicio():
    narrar(
        "Você acorda suando frio.",
        "O papel de parede floral está descascado nas bordas, "
        "como se alguém tivesse arrancado pétalas com as unhas. "
        "O colchão cheira a mofo e a sabão em pó antigo — o mesmo "
        "cheiro da sua infância.",
        "No espelho rachado do guarda-roupa, o reflexo pisca "
        "um segundo depois de você. Só um segundo. Mas o bastante "
        "para o estômago apertar.",
        "Na mesinha de cabeceira, um bilhete na sua letra:",
        '"Não abra a porta se ela já estiver aberta."',
        "A porta do quarto já está aberta. Além dela, o corredor "
        "respira uma luz amarela fraca.",
    )
    op = escolher(
        "1) Levantar e ir ao corredor\n2) Olhar o bilhete de novo\n> ",
        ["1", "2"],
    )
    if op == "2":
        state["leu_bilhete"] = True
        narrar(
            "Você pega o bilhete. O papel está úmido, como se "
            "tivesse acabado de ser escrito. No verso, em letra "
            "menor, trêmula:",
            '"Ela conta até quinze. Depois, a casa escolhe por você."',
            "O reflexo no espelho rachado agora está imóvel — "
            "demais. Você larga o bilhete e atravessa a porta.",
        )
    else:
        narrar(
            "Você se levanta. O piso range sob o pé esquerdo, "
            "depois — meio segundo depois — range de novo, sozinho. "
            "Você atravessa a porta aberta.",
        )
    return "corredor"


def corredor():
    if state["vida"] <= 0:
        return "fim_morte"
    narrar(
        "O corredor é estreito demais para uma casa. As paredes "
        "parecem ter se aproximado com os anos.",
        "Uma lâmpada amarela treme no teto. Longe, passos imitam "
        "os seus com meio segundo de atraso — clic… clic.",
        "À esquerda: a cozinha. À direita: a sala. No fundo, "
        "uma escada sobe para o sótão e desce para o porão.",
    )
    op = escolher(
        "1) Cozinha\n2) Sala\n3) Sótão\n4) Porão\n5) Chamar quem está aí\n> ",
        ["1", "2", "3", "4", "5"],
    )
    if op == "1":
        return "cozinha"
    if op == "2":
        return "sala"
    if op == "3":
        return "sotao"
    if op == "4":
        return _tentar_porao()
    return _chamar()


def _tentar_porao():
    if tem("chave_enferrujada"):
        if not state["abriu_porao"]:
            state["abriu_porao"] = True
            narrar(
                "A chave enferrujada gira com um queixume metálico. "
                "O ar que sobe do porão é úmido, doce e podre — "
                "como fruta esquecida no escuro.",
            )
        return "porao"
    narrar(
        "A porta do porão está trancada. A fechadura é antiga, "
        "coberta de ferrugem em forma de unha.",
    )
    op = escolher("1) Forçar a porta\n2) Desistir e voltar\n> ", ["1", "2"])
    if op == "1":
        state["forcou_porao"] = True
        narrar(
            "Você empurra com o ombro. A madeira geme, mas não cede. "
            "Algo do outro lado empurra de volta — no mesmo ritmo. "
            "Uma lasca corta sua palma.",
        )
        if perder_vida(1):
            return "fim_morte"
        narrar("Você recua, ofegante. O corredor espera.")
        return "corredor"
    narrar("Você se afasta. Os passos atrasados continuam, pacientes.")
    return "corredor"


def _chamar():
    narrar(
        'Você engole seco e grita: "Tem alguém aí?"',
        "O silêncio engorda. Então, do fundo do corredor, a sua "
        "própria voz responde — um pouco mais baixa, um pouco "
        "mais alegre:",
        '"Tem alguém aí?"',
        "Os passos atrasados aceleram. Algo frio roça sua nuca.",
    )
    state["conheceu_eco"] = True
    if perder_vida(1):
        return "fim_morte"
    narrar(
        "Quando você se vira, não há ninguém. Só a lâmpada "
        "tremendo mais forte.",
    )
    return "corredor"


def cozinha():
    narrar(
        "A cozinha cheira a gás antigo e laranja podre.",
        "Na pia, um prato com comida ainda quente — arroz, feijão, "
        "um pedaço de carne. O vapor sobe em espirais lentas. "
        "Ninguém mora aqui há doze anos.",
        "Na parede acima da mesa, riscos profundos formam letras "
        "tortas: CONTANDO ATÉ QUINZE.",
    )
    if tem("fosforos"):
        narrar(
            "A gaveta da esquerda está aberta e vazia. Você já "
            "pegou os fósforos.",
        )
        escolher("1) Voltar ao corredor\n> ", ["1"])
        return "corredor"
    narrar(
        "Na gaveta da esquerda, uma caixa de fósforos. A etiqueta "
        "está apagada, mas você lembra da marca — a mesma que "
        "seu pai usava para acender o fogão.",
    )
    op = escolher(
        "1) Pegar os fósforos\n2) Deixar e voltar ao corredor\n"
        "3) Tocar a comida quente\n> ",
        ["1", "2", "3"],
    )
    if op == "1":
        pegar("fosforos")
        narrar(
            "Você guarda os fósforos. A caixa é leve demais — "
            "quase vazia. Dentro, restam três palitos. Três chances.",
        )
    elif op == "3":
        narrar(
            "Você encosta o dedo no arroz. Escaldante. Quando "
            "retira a mão, a comida ainda fumega… e, por um "
            "instante, você vê uma segunda mão — a sua, mais "
            "nova — fazendo o mesmo gesto do outro lado do vapor.",
            "A visão some. O prato continua lá.",
        )
        op2 = escolher(
            "1) Pegar os fósforos agora\n2) Voltar ao corredor sem eles\n> ",
            ["1", "2"],
        )
        if op2 == "1":
            pegar("fosforos")
            narrar("Você pega os fósforos. A caixa treme na sua mão.")
        else:
            narrar("Você deixa a cozinha. O cheiro de laranja te segue.")
    else:
        narrar("Você fecha a gaveta sem pegar nada e volta.")
    return "corredor"


def sala():
    narrar(
        "A sala está coberta. O sofá usa um lençol branco manchado "
        "de amarelo. Relógios parados. Poeira em camadas.",
        "A TV de tubo olha para você com a tela morta — um olho "
        "cinza, opaco.",
    )
    if state["ouviu_fita"]:
        narrar(
            "A TV liga sozinha. Estática. No meio do ruído branco, "
            "um rosto se forma — o seu, mais novo, sorrindo sem "
            "chegar aos olhos.",
        )
    opcoes = ["1", "2", "3"]
    prompt = (
        "1) Abrir a gaveta da estante\n"
        "2) Examinar a TV\n"
        "3) Voltar ao corredor\n"
    )
    if state["ouviu_fita"] and state["viu_foto"]:
        prompt += "4) Seguir o rosto na estática (para o espelho)\n"
        opcoes.append("4")
    op = escolher(prompt + "> ", opcoes)
    if op == "1":
        return _gaveta()
    if op == "2":
        return _tv()
    if op == "4":
        narrar(
            "O rosto na estática inclina a cabeça. A tela estala. "
            "Você sente um puxão atrás dos olhos — e o mundo "
            "vira escuro úmido. Quando a visão volta, você está "
            "diante de um espelho coberto por um pano.",
        )
        return "espelho"
    narrar("Você deixa a sala. O lençol do sofá se mexe sem vento.")
    return "corredor"


def _gaveta():
    if tem("chave_enferrujada"):
        narrar("A gaveta está vazia. A chave já está com você.")
        return "sala"
    narrar(
        "A gaveta range. Dentro: uma chave enferrujada, quente "
        "ao toque, como se alguém a tivesse segurado agora há pouco. "
        "Na alça, um pedaço de fita isolante com a palavra PORÃO.",
    )
    pegar("chave_enferrujada")
    narrar("Você guarda a chave. Ela esfria na sua mão aos poucos.")
    return "sala"


def _tv():
    state["viu_tv"] = True
    if state["ouviu_fita"]:
        narrar(
            "Você se aproxima da tela. O rosto de criança abre "
            "a boca. Não sai som — sai um sopro frio pela fresta "
            "do painel. A estática sussurra:",
            '"Você deixou alguém no seu lugar."',
            "A coragem sobe na garganta como náusea. Agora você "
            "sabe: precisa ver o espelho de verdade.",
        )
    else:
        narrar(
            "Você liga a TV. Só estática. No chiado, quase dá "
            "para ouvir alguém contar: um… dois… três… "
            "Você desliga antes de chegar a quinze.",
        )
    return "sala"


def sotao():
    narrar(
        "A escada range a cada degrau. O sótão cheira a poeira "
        "quente e a madeira velha. Caixas empilhadas. Um baú "
        "de brinquedos cobertos por lençol.",
    )
    if not tem("fosforos"):
        return _sotao_sem_luz()
    return _sotao_com_luz()


def _sotao_sem_luz():
    narrar(
        "Está escuro demais. Suas mãos encontram arestas, "
        "teias, algo macio que pode ser um casaco… ou não.",
    )
    op = escolher("1) Insistir no escuro\n2) Descer ao corredor\n> ", ["1", "2"])
    if op == "2":
        narrar("Você desce. A escuridão do sótão parece aliviada.")
        return "corredor"
    narrar(
        "Você avança. O pé encontra o vazio entre duas tábuas. "
        "Você cai de joelho. Algo — uma unha? um fio? — "
        "raspa seu tornozelo.",
    )
    if perder_vida(1):
        return "fim_morte"
    narrar(
        "Você engatinha de volta à escada, o coração batendo "
        "atrasado, como os passos do corredor.",
    )
    return "corredor"


def _sotao_com_luz():
    narrar(
        "Você risca um fósforo. A chama treme e revela o sótão "
        "em pedaços laranja: caixas, um gravador de fita cassete, "
        "uma vela branca sem usar.",
    )
    while True:
        opcoes = ["1", "2", "3"]
        prompt = "1) Examinar o gravador / a fita\n"
        if not tem("vela"):
            prompt += "2) Pegar a vela\n"
        else:
            prompt += "2) (Você já tem a vela)\n"
        prompt += "3) Descer ao corredor\n"
        if state["ouviu_fita"] and (state["viu_foto"] or state["viu_tv"]):
            prompt += "4) Seguir o eco da fita (para o espelho)\n"
            opcoes.append("4")
        op = escolher(prompt + "> ", opcoes)
        if op == "1":
            _fita()
            continue
        if op == "2":
            if not tem("vela"):
                pegar("vela")
                narrar(
                    "Você guarda a vela. A cera está fria, mas "
                    "o pavio cheira a fumaça recente — "
                    "como se alguém tivesse apagado agora.",
                )
            else:
                narrar("A vela já está com você.")
            continue
        if op == "4":
            narrar(
                "A voz da criança na fita parece vir de baixo. "
                "Você segue o som escada abaixo, atravessa o "
                "corredor sem olhar, e desce ao porão — "
                "até um espelho coberto por pano.",
            )
            return "espelho"
        narrar("Você desce. O fósforo se apaga no último degrau.")
        return "corredor"


def _fita():
    if not tem("fita_cassete"):
        pegar("fita_cassete")
        narrar(
            "Você puxa a fita do gravador. A etiqueta, na sua "
            "letra de criança: EU / OUTRO.",
        )
    narrar(
        "Você aperta play. Chiado. Então uma voz de criança — "
        "a sua — sussurra perto demais do microfone:",
        '"Quando eu crescer, vou deixar alguém no meu lugar."',
        "Pausa. Respiração. Depois, mais baixo:",
        '"Pra casa não ficar sozinha."',
        "A fita termina. O gravador continua quente.",
    )
    state["ouviu_fita"] = True


def porao():
    narrar(
        "O porão engole o som. Umidade cola na pele. Nas paredes, "
        "marcas de unha formam o seu nome — letra por letra, "
        "profundas demais para uma brincadeira.",
    )
    if not tem("fosforos"):
        narrar(
            "Sem luz, o chão some. Você tropeça numa caixa. "
            "O joelho bate no concreto.",
        )
        if perder_vida(1):
            return "fim_morte"
        narrar(
            "Você engatinha até a escada, guiado só pelo cheiro "
            "menos podre de cima.",
        )
        op = escolher(
            "1) Subir ao corredor\n2) Tentar de novo no escuro\n> ",
            ["1", "2"],
        )
        if op == "1":
            return "corredor"
        narrar(
            "Você insiste. Dedos encontram um pano grosso sobre "
            "algo liso — vidro. Um espelho. Sem luz, você não "
            "ousa puxar o pano.",
        )
        if perder_vida(1):
            return "fim_morte"
        return "corredor"
    return _porao_com_luz()


def _porao_com_luz():
    narrar(
        "Você risca um fósforo. A chama mostra a inscrição "
        "completa nas paredes: o seu nome, repetido, e abaixo:",
        '"ELE FICOU."',
        "No chão, uma foto rasgada. No fundo, uma porta baixa "
        "coberta por um pano escuro — o formato de um espelho "
        "de corpo inteiro.",
    )
    while True:
        if not tem("foto_rasgada"):
            prompt = "1) Pegar a foto rasgada\n"
        else:
            prompt = "1) (Você já tem a foto)\n"
        prompt += "2) Puxar o pano do espelho\n3) Subir ao corredor\n"
        op = escolher(prompt + "> ", ["1", "2", "3"])
        if op == "1":
            if not tem("foto_rasgada"):
                pegar("foto_rasgada")
                state["viu_foto"] = True
                narrar(
                    "A foto mostra a casa intacta, ensolarada. "
                    "No jardim, uma criança — você — sorri. "
                    "Atrás dela, uma sombra com o mesmo sorriso, "
                    "atrasada um passo. A borda da foto está "
                    "queimada.",
                )
            else:
                narrar("Você já guarda a foto. O sorriso da sombra não muda.")
            continue
        if op == "2":
            narrar(
                "Você puxa o pano. O tecido cai como pele morta. "
                "O espelho não mostra o porão — mostra o corredor "
                "de cima, vazio… e alguém com o seu rosto "
                "já te esperando do outro lado, sorrindo.",
            )
            state["conheceu_eco"] = True
            return "espelho"
        narrar("Você sobe. O fósforo morre entre os dedos.")
        return "corredor"


def espelho():
    if state["vida"] <= 0:
        return "fim_morte"
    tem_pista = (
        state["ouviu_fita"]
        or state["viu_foto"]
        or tem("fita_cassete")
        or tem("foto_rasgada")
    )
    if not tem_pista:
        narrar(
            "O espelho te engole com a própria imagem. Sem "
            "lembrar por que veio, você só vê o sorriso atrasado. "
            "Mãos iguais às suas atravessam o vidro e puxam.",
        )
        return "fim_morte"
    narrar(
        "O doppelgänger está do outro lado do vidro, sorrindo "
        "com o seu sorriso. Ele fala primeiro — com a sua voz, "
        "meio segundo atrasada:",
        '"Eu esperei doze anos. A casa estava com saudade."',
        "O ar cheira a chuva de infância e a ferrugem.",
    )
    state["conheceu_eco"] = True
    opcoes = ["1", "2", "3"]
    prompt = (
        "1) Correr para a porta da frente\n"
        '2) Confrontar: "Você não é eu"\n'
        "3) Aceitar trocar de lugar\n"
    )
    if tem("vela") and tem("fosforos") and (
        state["ouviu_fita"] or tem("fita_cassete")
    ):
        prompt += "4) Acender a vela e dizer o nome do bilhete\n"
        opcoes.append("4")
    op = escolher(prompt + "> ", opcoes)
    if op == "1":
        if tem("chave_enferrujada") or tem("fosforos"):
            return "fim_fuga"
        narrar(
            "Você corre. Sem chave, sem luz. A porta da frente "
            "está trancada por dentro. Atrasado, o eco chega "
            "e põe a mão no seu ombro — a mesma mão.",
        )
        return "fim_morte"
    if op == "2":
        if (state["ouviu_fita"] or tem("fita_cassete")) and (
            state["viu_foto"] or tem("foto_rasgada")
        ):
            return "fim_verdade"
        narrar(
            'Você grita: "Você não é eu!" O eco ri com a sua '
            "garganta. Sem a fita e a foto, a frase não tem peso. "
            "O vidro não quebra. Você quebra.",
        )
        return "fim_morte"
    if op == "3":
        return "fim_eco"
    if tem("vela") and tem("fosforos") and (
        state["ouviu_fita"] or tem("fita_cassete")
    ):
        state["apagou_vela"] = True
        return "fim_ritual"
    return "fim_morte"


def _resumo(titulo):
    print("---")
    print(f"Final: {titulo}")
    print(status_linha())
    print("---")


def fim_fuga():
    narrar(
        "Você corre. A porta da frente cede — chave ou luz, "
        "não importa. A noite lá fora é real: vento, rua, "
        "o cheiro de asfalto molhado.",
        "Você olha para trás. A casa está quieta. Então, "
        "no seu antigo quarto, a luz acende sozinha.",
        "Alguém passa atrás da cortina com o seu jeito de andar. "
        "Meio segundo atrasado.",
        "Você escapou. Talvez.",
    )
    _resumo("FUGA AMBÍGUA")
    return "fim"


def fim_verdade():
    narrar(
        'Você segura a foto e a memória da fita. "Você não é eu. '
        'Você é o que eu deixei."',
        "O sorriso do eco trinca. Você golpeia o espelho. "
        "O vidro soluça — um som úmido, humano — e estilhaça.",
        "A casa inteira respira fundo, como quem larga um "
        "segredo. Poeira sobe. A luz amarela morre.",
        "Quando amanhece, você está no jardim. A porta está "
        "fechada. Não há passos atrasados. Só o seu coração, "
        "no tempo certo.",
    )
    _resumo("VERDADE")
    return "fim"


def fim_eco():
    narrar(
        "Você encosta a mão no vidro. O eco encosta a dele. "
        "O frio passa. O calor fica do outro lado.",
        "Você tenta recuar. Não há recuo. O porão — ou a sala, "
        "ou o quarto — agora é o lado de dentro do espelho.",
        "Do lado de fora, alguém com o seu rosto abre a porta "
        "da frente, inspira a noite e sorri no tempo certo.",
        "A casa não está mais sozinha. Você está.",
    )
    _resumo("O ECO SAI")
    return "fim"


def fim_ritual():
    narrar(
        "Você risca o fósforo. A vela acende. A chama mostra "
        "o eco como ele é: pequeno, assustado, uma criança "
        "que prometeu não deixar a casa vazia.",
        "Você diz o que o bilhete escondia — não um nome de "
        "pessoa, mas o nome da casa, o apelido que só vocês "
        "dois usavam quando era tarde demais para dormir.",
        "O eco encolhe. Vira menino de novo. A vela tremula. "
        "Você apaga com os dedos.",
        "De manhã, você tranca cada porta. No quintal, queima "
        "a fita. A fumaça sobe reta. A casa, pela primeira vez "
        "em doze anos, não responde.",
    )
    _resumo("RITUAL — FINAL SECRETO")
    return "fim"


def fim_morte():
    narrar(
        "A escuridão fecha como uma boca.",
        "Os passos atrasados — clic… clic — param.",
        "Não porque foram embora.",
        "Porque agora estão sincronizados com os seus.",
        "A casa lembra. E você, enfim, também.",
    )
    _resumo("MORTE")
    return "fim"


def fim_atrasado():
    narrar(
        "Quinze. A casa completa a contagem por você.",
        "Do corredor vem a sua voz, paciente, quase carinhosa:",
        '"Não abra a porta se ela já estiver aberta."',
        "A porta do quarto — aquela que você deixou para trás — "
        "fecha por dentro. A chave gira sozinha.",
        "O eco completa a frase do bilhete no seu ouvido:",
        '"Agora eu abro."',
        "Não há mais escolhas. Só a casa, lembrando.",
    )
    _resumo("ATRASADO DEMAIS")
    return "fim"


CENAS = {
    "inicio": inicio,
    "corredor": corredor,
    "cozinha": cozinha,
    "sala": sala,
    "sotao": sotao,
    "porao": porao,
    "espelho": espelho,
    "fim_fuga": fim_fuga,
    "fim_verdade": fim_verdade,
    "fim_eco": fim_eco,
    "fim_ritual": fim_ritual,
    "fim_morte": fim_morte,
    "fim_atrasado": fim_atrasado,
}

FINAIS = {
    "fim_fuga", "fim_verdade", "fim_eco",
    "fim_ritual", "fim_morte", "fim_atrasado",
}


def jogar():
    """Inicia (ou reinicia) uma partida."""
    resetar()
    print("=" * 48)
    print("  A CASA QUE LEMBRA")
    print("  Um jogo de terror em texto")
    print("=" * 48)
    print(
        f"\nVocê tem no máximo {MAX_TURNOS} interações. "
        "Cada escolha conta.\nA casa está contando junto com você.\n"
    )
    cena = "inicio"
    while cena != "fim":
        if state["turnos"] >= MAX_TURNOS and cena not in FINAIS:
            cena = "fim_atrasado"
            continue
        if cena not in CENAS:
            print(f"Cena desconhecida: {cena}")
            break
        cena = CENAS[cena]()
    print("\nFim.\n")
    print("Para jogar de novo, execute de novo: jogar()")


print("Jogo carregado. Execute a próxima célula: jogar()")



## Jogar

Rode a célula abaixo. Quando aparecer `>`, digite `1`, `2`, etc. e Enter.


In [ ]:
jogar()
